In [14]:
%env DB_PASSWORD=Wtcantfw36c!

env: DB_PASSWORD=Wtcantfw36c!


In [31]:
import pandas as pd
import os
from sqlalchemy import create_engine
from constants.misc_constants import AUTO_GENERATED_COLUMNS

In [16]:
pw = os.environ.get("DB_PASSWORD")
SQL_ALCH_CONFIG = {
        'host': 'dandyweb01fl',
        'database': 'aedna_metadata_test',
        'port': '5432',
        'user': 'upload_user',
        'password': pw,
        'schema_name': 'test_1'
    }
DATABASE_CONFIG_READ_ONLY = {
    'host': SQL_ALCH_CONFIG['host'],
    'dbname': SQL_ALCH_CONFIG['database'],
    'port': SQL_ALCH_CONFIG['port'],
    'user': 'read_user',
    'password': SQL_ALCH_CONFIG['password'],
}
ENGINE_READ_ONLY = create_engine(
    f"postgresql://{DATABASE_CONFIG_READ_ONLY['user']}:{DATABASE_CONFIG_READ_ONLY['password']}@{SQL_ALCH_CONFIG['host']}:{SQL_ALCH_CONFIG['port']}/{SQL_ALCH_CONFIG['database']}")


In [44]:
query = '''
select * from test_1.field_sample fs2 
full outer join test_1.edna_archive_sample eas on lower(fs2.field_sample_id) = lower(eas.field_sample_id)
full outer join test_1.edna_robot_sample ers on lower(eas.archive_sample_id) = lower(ers.archive_sample_id)
full outer join test_1.edna_wetlab_report ewr on lower(ers.robot_sample_id) = lower(ewr.robot_sample_id) 
full outer join test_1.flowcell f on lower(ewr.fastq_file_id) = lower(f.fastq_file_id);
'''



fs = pd.read_sql('select * from test_1.field_sample', ENGINE_READ_ONLY).drop(columns=AUTO_GENERATED_COLUMNS, errors="ignore")
as_ = pd.read_sql('select * from test_1.edna_archive_sample', ENGINE_READ_ONLY).drop(columns=AUTO_GENERATED_COLUMNS, errors="ignore")
rs = pd.read_sql('select * from test_1.edna_robot_sample', ENGINE_READ_ONLY).drop(columns=AUTO_GENERATED_COLUMNS, errors="ignore")
wl = pd.read_sql('select * from test_1.edna_wetlab_report', ENGINE_READ_ONLY).drop(columns=AUTO_GENERATED_COLUMNS, errors="ignore")
fc = pd.read_sql('select * from test_1.flowcell', ENGINE_READ_ONLY).drop(columns=AUTO_GENERATED_COLUMNS, errors="ignore")
full_query = pd.read_sql(query, ENGINE_READ_ONLY).drop(columns=AUTO_GENERATED_COLUMNS, errors="ignore")


In [59]:
full_df = (
    fs.merge(as_, how="outer", on='field_sample_id')
    .merge(rs, "outer", on='archive_sample_id')
    .merge(wl, "outer", on='robot_sample_id')
    .merge(fc, how="outer", on='fastq_file_id')
)

In [54]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [61]:
len(full_df[full_df['flowcell_id'] == "H5FH3DSX7"])

168

In [50]:
full_query.columns.value_counts()

archive_sample_id                    3
field_sample_id                      2
robot_sample_id                      2
fastq_file_id                        2
RackID                               2
                                    ..
Site-Grid Elev (m asl)               1
Sample Provider(s) (Contact info)    1
PI (Full name)                       1
Other Relevant Information           1
Yield (MBases) sum                   1
Name: count, Length: 119, dtype: int64

In [48]:
for col in full_query.columns:
    print(col)

field_sample_id
country_ocean
Site name
Latitude (WGS84 decimal degrees)
Longitude (WGS84 decimal degrees)
Sample date
Sample Provider(s) (Full name)
Running Project Title
Sampling depth (discrete, cm)
Sampling interval - to (cm)
Water depth (m)
Sample type
Sample environment
Sample context
Age Estimate - from (ka)
Elevation (m asl)
Sample storage address
Sample storage setting
Sample storage location
Miscellaneous Sample Measurement(s) or Observation(s)
Miscellaneous Environmental Measurement(s) or Observation(s)
Link(s) to images
Link(s) to other relevant information
Comments
Sampling interval - from (cm)
Age Estimate - to (ka)
Sample material
Alias
Cultural Affiliation
Museum/Institution
Other Relevant Information
PI (Full name)
Sample Provider(s) (Contact info)
Site-Grid Elev (m asl)
Site-Grid Latitude (WGS84 decimal degrees)
Site-Grid Longitude (WGS84 decimal degrees)
field_sample_parent_id
Field Label (informal)
Sample type in storage at GM
Permit for DNA Analysis (yes/no)
archiv

In [40]:
len(full_df)

33684

In [22]:
len(df.columns)

152

In [24]:
df = df.drop(columns=["database_insert_datetime_utc", "upload_uuid", "from_spreadsheet", "database_insert_by"])

In [25]:
len(df.columns)

132